# Preprocessing — Telco Customer Churn

## Ce facem in acest notebook
1. **Curatare**: spatii ascunse, valori lipsa, outlieri, duplicate, text inconsistent
2. **Encoding**: transformam textul in numere (get_dummies)
3. **Scalare**: aducem coloanele numerice la aceeasi scara
4. **Train/Test Split**: impartim datele 80/20
5. **SMOTE**: echilibram clasele (27% → 50%)

## Date messy introduse intentionat (pentru practica)
- 10 spatii goale in TotalCharges
- 8 valori NaN in tenure
- 5 valori negative (-10) in tenure
- 3 outlieri (9999) in MonthlyCharges
- 3 randuri duplicate
- Text inconsistent in Churn (Yes/yes/YES)

In [176]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import os

In [177]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv',sep=',')
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Putem sa tragem urmatoare concluzii:

1.TotalCharges trebuie sa fie float64

2.Avem 7043 linii | 21 coloane

3.Coloana Churn trebuie transformata in valoare numerica 

In [178]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [179]:
#ultimul_rand
rand=df.iloc[[0]]

df.loc[50:59, 'TotalCharges'] = ' '

df.loc[300:304, 'tenure'] = -10

df.loc[234,'MonthlyCharges'] = 9999
df.loc[500,'MonthlyCharges'] = 9999
df.loc[1200,'MonthlyCharges'] = 9999

#cum fac asta de 3 ori ? 
df=pd.concat([df,rand,rand,rand], ignore_index = True)

df.loc[400:402,'Churn']='yes'
df.loc[600,'Churn'] = 'YES'

df.loc[100:107, 'tenure'] = np.nan



In [180]:
print(df.shape)
print(df.info())
print(df['Churn'].unique())
print(df['MonthlyCharges'].dtype)

(7046, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7046 entries, 0 to 7045
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7046 non-null   str    
 1   gender            7046 non-null   str    
 2   SeniorCitizen     7046 non-null   int64  
 3   Partner           7046 non-null   str    
 4   Dependents        7046 non-null   str    
 5   tenure            7038 non-null   float64
 6   PhoneService      7046 non-null   str    
 7   MultipleLines     7046 non-null   str    
 8   InternetService   7046 non-null   str    
 9   OnlineSecurity    7046 non-null   str    
 10  OnlineBackup      7046 non-null   str    
 11  DeviceProtection  7046 non-null   str    
 12  TechSupport       7046 non-null   str    
 13  StreamingTV       7046 non-null   str    
 14  StreamingMovies   7046 non-null   str    
 15  Contract          7046 non-null   str    
 16  PaperlessBilling  7046 non-null   str    


# 2.1 Inlocuim spatiile ascunse cu NaN real



In [181]:
df = df.replace([' ', '', 'N/A', 'NULL', '-', '?'], np.nan)
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7046 entries, 0 to 7045
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7046 non-null   str    
 1   gender            7046 non-null   str    
 2   SeniorCitizen     7046 non-null   int64  
 3   Partner           7046 non-null   str    
 4   Dependents        7046 non-null   str    
 5   tenure            7038 non-null   float64
 6   PhoneService      7046 non-null   str    
 7   MultipleLines     7046 non-null   str    
 8   InternetService   7046 non-null   str    
 9   OnlineSecurity    7046 non-null   str    
 10  OnlineBackup      7046 non-null   str    
 11  DeviceProtection  7046 non-null   str    
 12  TechSupport       7046 non-null   str    
 13  StreamingTV       7046 non-null   str    
 14  StreamingMovies   7046 non-null   str    
 15  Contract          7046 non-null   str    
 16  PaperlessBilling  7046 non-null   str    
 17  Paymen

# 2.2 TotalCharges: text -> numeric


In [182]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7046 entries, 0 to 7045
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7046 non-null   str    
 1   gender            7046 non-null   str    
 2   SeniorCitizen     7046 non-null   int64  
 3   Partner           7046 non-null   str    
 4   Dependents        7046 non-null   str    
 5   tenure            7038 non-null   float64
 6   PhoneService      7046 non-null   str    
 7   MultipleLines     7046 non-null   str    
 8   InternetService   7046 non-null   str    
 9   OnlineSecurity    7046 non-null   str    
 10  OnlineBackup      7046 non-null   str    
 11  DeviceProtection  7046 non-null   str    
 12  TechSupport       7046 non-null   str    
 13  StreamingTV       7046 non-null   str    
 14  StreamingMovies   7046 non-null   str    
 15  Contract          7046 non-null   str    
 16  PaperlessBilling  7046 non-null   str    
 17  Paymen

In [183]:
print(df.shape)
print(df.info())
print(df['Churn'].unique())
print(df['MonthlyCharges'].dtype)

(7046, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7046 entries, 0 to 7045
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7046 non-null   str    
 1   gender            7046 non-null   str    
 2   SeniorCitizen     7046 non-null   int64  
 3   Partner           7046 non-null   str    
 4   Dependents        7046 non-null   str    
 5   tenure            7038 non-null   float64
 6   PhoneService      7046 non-null   str    
 7   MultipleLines     7046 non-null   str    
 8   InternetService   7046 non-null   str    
 9   OnlineSecurity    7046 non-null   str    
 10  OnlineBackup      7046 non-null   str    
 11  DeviceProtection  7046 non-null   str    
 12  TechSupport       7046 non-null   str    
 13  StreamingTV       7046 non-null   str    
 14  StreamingMovies   7046 non-null   str    
 15  Contract          7046 non-null   str    
 16  PaperlessBilling  7046 non-null   str    


# 2.5 Tratare outlieri (daca exista)


In [184]:
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: {outliers} outlieri (range: {lower:.1f} - {upper:.1f})")

tenure: 0 outlieri (range: -60.0 - 124.0)
MonthlyCharges: 3 outlieri (range: -46.0 - 171.4)
TotalCharges: 0 outlieri (range: -4677.0 - 8860.6)


In [185]:
median = df['MonthlyCharges'].median()
df.loc[df['MonthlyCharges'] > median*5, 'MonthlyCharges'] = median
print(df['MonthlyCharges'].max())

118.75


In [186]:
df[df['tenure']<0]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
300,0895-LMRSF,Male,0,No,No,-10.0,Yes,No,DSL,No,...,No,No,Yes,Yes,One year,Yes,Bank transfer (automatic),64.90,1509.80,No
301,8098-LLAZX,Female,1,No,No,-10.0,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.45,396.10,Yes
302,8266-VBFQL,Male,0,No,No,-10.0,Yes,No,Fiber optic,No,...,Yes,Yes,No,Yes,Month-to-month,Yes,Electronic check,90.40,356.65,No
303,8181-YHCMF,Female,0,Yes,Yes,-10.0,No,No phone service,DSL,No,...,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),60.30,4109.00,No
304,2240-HSJQD,Male,0,No,Yes,-10.0,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,81.85,3141.70,No


In [187]:
df.loc[300:305,'tenure']=df['tenure'].median()
print(df[df['tenure'] < 0].shape[0])
#sau
#df.loc[df['tenure'] < 0, 'tenure'] = df['tenure'].median()

0


In [188]:
df['tenure'] =df['tenure'].fillna(df['tenure'].median())
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7046 entries, 0 to 7045
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7046 non-null   str    
 1   gender            7046 non-null   str    
 2   SeniorCitizen     7046 non-null   int64  
 3   Partner           7046 non-null   str    
 4   Dependents        7046 non-null   str    
 5   tenure            7046 non-null   float64
 6   PhoneService      7046 non-null   str    
 7   MultipleLines     7046 non-null   str    
 8   InternetService   7046 non-null   str    
 9   OnlineSecurity    7046 non-null   str    
 10  OnlineBackup      7046 non-null   str    
 11  DeviceProtection  7046 non-null   str    
 12  TechSupport       7046 non-null   str    
 13  StreamingTV       7046 non-null   str    
 14  StreamingMovies   7046 non-null   str    
 15  Contract          7046 non-null   str    
 16  PaperlessBilling  7046 non-null   str    
 17  Paymen

# TotalCharges: cu tenure * MonthlyCharges (logica de business)


In [189]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'])
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [190]:
df.duplicated().sum()

np.int64(3)

In [191]:
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   float64
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [192]:
print(df['Churn'].unique())


<ArrowStringArray>
['No', 'Yes', 'yes', 'YES']
Length: 4, dtype: str


In [193]:
df['Churn'] = df['Churn'].str.strip().str.lower()

In [194]:
print(df.select_dtypes(include='object').columns.tolist())

['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


In [195]:
# Scoatem customerID (nu ajuta modelul)
df = df.drop('customerID', axis=1)

# get_dummies pe toate coloanele text
df = pd.get_dummies(df, drop_first=True, dtype=int)

print(f"Shape dupa encoding: {df.shape}")
print(f"\nColoane: {df.columns.tolist()}")

Shape dupa encoding: (7043, 31)

Coloane: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'Churn_yes']


In [196]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7043 non-null   int64  
 1   tenure                                 7043 non-null   float64
 2   MonthlyCharges                         7043 non-null   float64
 3   TotalCharges                           7043 non-null   float64
 4   gender_Male                            7043 non-null   int64  
 5   Partner_Yes                            7043 non-null   int64  
 6   Dependents_Yes                         7043 non-null   int64  
 7   PhoneService_Yes                       7043 non-null   int64  
 8   MultipleLines_No phone service         7043 non-null   int64  
 9   MultipleLines_Yes                      7043 non-null   int64  
 10  InternetService_Fiber optic            7043 non-null   int64  
 11  InternetService

## 4. Scalare

In [197]:
# Identificam coloanele care trebuie scalate (float cu valori mari)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print("Inainte de scalare:")
print(df[num_cols].describe().round(2))

Inainte de scalare:
        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7043.00
mean     32.36           64.76       2279.73
std      24.53           30.08       2266.78
min       0.00           18.25          0.00
25%       9.00           35.52        398.55
50%      29.00           70.35       1394.55
75%      55.00           89.85       3786.60
max      72.00          118.75       8684.80


In [198]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
num_cols=df.select_dtypes(include='float64').columns.tolist()
df[num_cols]=scaler.fit_transform(df[num_cols])

In [199]:
print(df[num_cols].describe().round(2))

        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7043.00
mean     -0.00           -0.00          0.00
std       1.00            1.00          1.00
min      -1.32           -1.55         -1.01
25%      -0.95           -0.97         -0.83
50%      -0.14            0.19         -0.39
75%       0.92            0.83          0.66
max       1.62            1.80          2.83


In [200]:
print(scaler.mean_)

[  32.36177765   64.75674429 2279.73252165]


In [201]:
import joblib
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']

Separam datele pe acre vrem sa le prezicem adica churn_yes de celelalte date

In [202]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn_yes', axis=1)  
y = df['Churn_yes']   

In [203]:
print([c for c in df.columns if 'churn' in c.lower()])

['Churn_yes']


In [204]:
X_train,X_test,y_train,y_test=train_test_split(
                                X,y,
                                test_size=0.2,
                                random_state=42,
                                stratify=y
                                )

In [205]:
print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")
print(f"Churn in train: {y_train.mean()*100:.1f}%")
print(f"Churn in test: {y_test.mean()*100:.1f}%")

Train: (5634, 30)
Test: (1409, 30)
Churn in train: 26.6%
Churn in test: 26.6%


In [206]:
# SMOTE creeaza exemple sintetice pentru clasa minoritara
# Se aplica DOAR pe train, NICIODATA pe test
print("Inainte de SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nDupa SMOTE:")
print(pd.Series(y_train_sm).value_counts())
print(f"\nTrain shape: {X_train_sm.shape}")

Inainte de SMOTE:
Churn_yes
0    4137
1    1497
Name: count, dtype: int64

Dupa SMOTE:
Churn_yes
1    4137
0    4137
Name: count, dtype: int64

Train shape: (8274, 30)


In [207]:
os.makedirs('../data/processed', exist_ok=True)

pd.DataFrame(X_train_sm, columns=X.columns).to_csv('../data/processed/X_train.csv',index=False)
pd.Series(y_train_sm, name='Churn_yes').to_csv('../data/processed/y_train.csv', index=False)

X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Salvam si numele coloanelor
joblib.dump(X.columns.tolist(), '../models/feature_names.pkl')

print("Date salvate in data/processed/")
print(f"  X_train: {X_train_sm.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  Features: {X.shape[1]}")


Date salvate in data/processed/
  X_train: (8274, 30)
  X_test:  (1409, 30)
  Features: 30
